# Graph Attention Networks

In [1]:
import numpy as np
np.random.seed(0)
A = np.array([
[1, 1, 1, 1],
[1, 1, 0, 0],
[1, 0, 1, 1],
[1, 0, 1, 1]
])
A

array([[1, 1, 1, 1],
       [1, 1, 0, 0],
       [1, 0, 1, 1],
       [1, 0, 1, 1]])

In [2]:
X = np.random.uniform(-1, 1, (4, 4))
X

array([[ 0.09762701,  0.43037873,  0.20552675,  0.08976637],
       [-0.1526904 ,  0.29178823, -0.12482558,  0.783546  ],
       [ 0.92732552, -0.23311696,  0.58345008,  0.05778984],
       [ 0.13608912,  0.85119328, -0.85792788, -0.8257414 ]])

In [8]:
W = np.random.uniform(-1,1,(2,4))
W_att = np.random.uniform(-1,1,(1,4))
W, W_att

(array([[ 0.04369664, -0.17067612, -0.47088878,  0.54846738],
        [-0.08769934,  0.1368679 , -0.9624204 ,  0.23527099]]),
 array([[0.22419145, 0.23386799, 0.88749616, 0.3636406 ]]))

In [9]:
connections = np.where(A > 0)
connections

(array([0, 0, 0, 0, 1, 1, 2, 2, 2, 3, 3, 3]),
 array([0, 1, 2, 3, 0, 1, 0, 2, 3, 0, 2, 3]))

In [10]:
s=np.concatenate([(X @ W.T)[connections[0]], (X @ W.T)[connections[1]]], axis=1)
s

array([[-0.11673572, -0.12634051, -0.11673572, -0.12634051],
       [-0.11673572, -0.12634051,  0.43205504,  0.35780762],
       [-0.11673572, -0.12634051, -0.16273574, -0.66116004],
       [-0.11673572, -0.12634051, -0.18823534,  0.7359804 ],
       [ 0.43205504,  0.35780762, -0.11673572, -0.12634051],
       [ 0.43205504,  0.35780762,  0.43205504,  0.35780762],
       [-0.16273574, -0.66116004, -0.11673572, -0.12634051],
       [-0.16273574, -0.66116004, -0.16273574, -0.66116004],
       [-0.16273574, -0.66116004, -0.18823534,  0.7359804 ],
       [-0.18823534,  0.7359804 , -0.11673572, -0.12634051],
       [-0.18823534,  0.7359804 , -0.16273574, -0.66116004],
       [-0.18823534,  0.7359804 , -0.18823534,  0.7359804 ]])

In [11]:
a = W_att @ s.T
a

array([[-0.20526319,  0.45784242, -0.44057013,  0.04485606,  0.03099776,
         0.69410336, -0.34065317, -0.57596011, -0.09053392, -0.01962353,
        -0.25493047,  0.23049572]])

In [12]:
def leaky_relu(x, alpha=0.2):
  return np.maximum(alpha*x, x)
e = leaky_relu(a)
e

array([[-0.04105264,  0.45784242, -0.08811403,  0.04485606,  0.03099776,
         0.69410336, -0.06813063, -0.11519202, -0.01810678, -0.00392471,
        -0.05098609,  0.23049572]])

In [13]:
E = np.zeros(A.shape)
E[connections[0], connections[1]] = e[0]
E

array([[-0.04105264,  0.45784242, -0.08811403,  0.04485606],
       [ 0.03099776,  0.69410336,  0.        ,  0.        ],
       [-0.06813063,  0.        , -0.11519202, -0.01810678],
       [-0.00392471,  0.        , -0.05098609,  0.23049572]])

In [14]:
def softmax2D(x, axis):
  e = np.exp(x - np.expand_dims(np.max(x, axis=axis),axis))
  sum = np.expand_dims(np.sum(e, axis=axis), axis)
  return e / sum
W_alpha = softmax2D(E, 1)
W_alpha

array([[0.2131907 , 0.35110387, 0.20339007, 0.23231536],
       [0.20492786, 0.39772613, 0.198673  , 0.198673  ],
       [0.24534879, 0.26264714, 0.23406982, 0.25793424],
       [0.23684688, 0.23777826, 0.22595875, 0.29941611]])

In [15]:
H = A.T @ W_alpha @ X @ W.T
H

array([[0.10831488, 0.49066432],
       [0.12816914, 0.26648272],
       [0.03012618, 0.35938079],
       [0.03012618, 0.35938079]])

## GATv2

In [21]:
!pip install torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 14.9 MB/s eta 0:00:00


In [1]:
from torch_geometric.datasets import Planetoid
dataset = Planetoid(root=".", name="Cora")
data = dataset[0]
data

/home/venv/trch/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Data(x=[2708, 1433], edge_index=[2, 10556], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708])

In [2]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GATv2Conv
from torch.nn import Linear, Dropout

In [3]:
def accuracy(y_pred, y_true):
  return torch.sum(y_pred == y_true) / len(y_true)

# CORA classes prediction

In [4]:
class GAT(torch.nn.Module):
  def __init__(self, dim_in, dim_h, dim_out, heads=8):
    super().__init__()
    self.gat1 = GATv2Conv(dim_in, dim_h, heads=heads)
    self.gat2 = GATv2Conv(dim_h*heads, dim_out,heads=1)
  def forward(self, x, edge_index):
    h = F.dropout(x, p=0.6, training=self.training)
    h = self.gat1(h, edge_index)
    h = F.elu(h)
    h = F.dropout(h, p=0.6, training=self.training)
    h = self.gat2(h, edge_index)
    return F.log_softmax(h, dim=1)
  def fit(self, data, epochs):
    criterion = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(self.parameters(),lr=0.01, weight_decay=0.01)
    self.train()
    for epoch in range(epochs+1):
      optimizer.zero_grad()
      out = self(data.x, data.edge_index)
      loss = criterion(out[data.train_mask],data.y[data.train_mask])
      acc = accuracy(out[data.train_mask].argmax(dim=1), data.y[data.train_mask])
      loss.backward()
      optimizer.step()
      if(epoch % 20 == 0):
        val_loss = criterion(out[data.val_mask],data.y[data.val_mask])
        val_acc = accuracy(out[data.val_mask].argmax(dim=1), data.y[data.val_mask])
        print(f'Epoch {epoch:>3} | Train Loss:{loss:.3f} | Train Acc: {acc*100:>5.2f}% | Val Loss:{val_loss:.2f} | Val Acc: {val_acc*100:.2f}%')
  @torch.no_grad()
  def test(self, data):
    self.eval()
    out = self(data.x, data.edge_index)
    acc = accuracy(out.argmax(dim=1)[data.test_mask],data.y[data.test_mask])
    return acc

gat = GAT(dataset.num_features, 32, dataset.num_classes)
gat.fit(data, epochs=100)

Epoch   0 | Train Loss:1.988 | Train Acc: 15.71% | Val Loss:1.99 | Val Acc: 10.60%
Epoch  20 | Train Loss:0.198 | Train Acc: 97.86% | Val Loss:0.86 | Val Acc: 74.60%
Epoch  40 | Train Loss:0.158 | Train Acc: 97.86% | Val Loss:0.81 | Val Acc: 74.40%
Epoch  60 | Train Loss:0.163 | Train Acc: 98.57% | Val Loss:0.79 | Val Acc: 75.00%
Epoch  80 | Train Loss:0.154 | Train Acc: 97.86% | Val Loss:0.85 | Val Acc: 75.20%
Epoch 100 | Train Loss:0.188 | Train Acc: 96.43% | Val Loss:0.94 | Val Acc: 72.00%


In [5]:
acc = gat.test(data)
print(f'GAT test accuracy: {acc*100:.2f}%')

GAT test accuracy: 80.20%
